In [5]:
import pandas as pd
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import pickle

# 1. Data
stocks = ["AAPL", "MSFT", "TSLA"]
df = yf.download(stocks, period="5y", group_by='ticker')

# 2. Combine
frames = []
for stock in stocks:
    temp = df[stock].copy()
    frames.append(temp)

df = pd.concat(frames)

# 3. Features
df['MA10'] = df['Close'].rolling(10).mean()
df['MA50'] = df['Close'].rolling(50).mean()
df['EMA10'] = df['Close'].ewm(span=10).mean()
df['Return'] = df['Close'].pct_change()
df['High_Low_Diff'] = df['High'] - df['Low']

# 4. Target (IMPORTANT CHANGE ✅)
df['Target'] = df['Close'].pct_change().shift(-1)

# Clean
df = df.dropna()

# 5. X and y
X = df[['Open','High','Low','Close','Volume',
        'MA10','MA50','EMA10','Return','High_Low_Diff']]
y = df['Target']

# 6. Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# 7. Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 8. Model
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

# 9. Evaluation
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print("MAE (Return):", mae)
print("Sample Predictions:", y_pred[:5])

# 10. Save
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)


[*********************100%***********************]  3 of 3 completed


MAE (Return): 0.028960157573348286
Sample Predictions: [ 0.00311726 -0.01295851 -0.00723852 -0.00940366 -0.0043566 ]
